In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [ ]:
from pyhydra.climate.time_series import (
    extract_discharge_events,
    extract_precipitation_events,
    extract_events,
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# ⛰️ Hydrological Event Extraction

## 📚 General Description

This notebook demonstrates how to automatically identify and characterise **hydrological events** from daily time series:

| Variable | Method | Basis |
|----------|--------|-------|
| **Discharge** | Inflection-point method | Rising / falling limb threshold crossings |
| **Precipitation** | Wet-spell method | Consecutive wet days with gap bridging |

---

## 📦 Output Format

Both extractors return two DataFrames:

**`stats_df`** — one row per event:

| Column | Description |
|--------|-------------|
| `peak` / `total` | Peak discharge (m³/s) or total rainfall (mm) |
| `mean` / `mean_intensity` | Mean value during event |
| `duration` | Number of days |
| `volume` | Volume (discharge events only) |
| `date_peak` / `date_start` | Date of peak or spell start |

**`bounds_df`** — event boundaries:

| Column | Description |
|--------|-------------|
| `start` | Start date |
| `end` | End date |

---
## 1. 🌊 Discharge Event Extraction

### 💡 Method: Inflection-point

1. Detect **rising crossings** of the primary threshold `threshold`
2. Find the first **falling crossing** of `threshold2` (or `threshold`) after each rise
3. Snap boundaries to the nearest local minimum (slope reversal)
4. Extract peak, mean discharge, duration, and runoff volume

In [ ]:
# Generate a synthetic multi-year hydrograph
rng   = np.random.default_rng(42)
dates = pd.date_range("2000-01-01", "2020-12-31", freq="D")
n     = len(dates)
t     = np.arange(n)

baseflow = 20 + 15 * np.sin(2 * np.pi * t / 365 - 1.0)
Q = baseflow + rng.exponential(3, n)

for _ in range(30):                              # add synthetic flood peaks
    i   = rng.integers(30, n - 30)
    pk  = rng.uniform(50, 200)
    dur = rng.integers(5, 20)
    rise = np.linspace(0, pk, dur // 2)
    fall = np.linspace(pk, 0, dur - dur // 2)
    sh   = np.concatenate([rise, fall])
    end  = min(i + len(sh), n)
    Q[i:end] += sh[:end - i]

Q_series = pd.Series(Q, index=dates, name="discharge")
print(f"Mean Q: {Q_series.mean():.1f} m³/s  |  Max Q: {Q_series.max():.1f} m³/s")

In [ ]:
# === PARAMETERS ===
THRESHOLD  = 60.0   # rising limb threshold (m³/s)
THRESHOLD2 = 45.0   # receding limb threshold (m³/s) — optional

stats_q, bounds_q = extract_discharge_events(
    series=Q_series,
    threshold=THRESHOLD,
    threshold2=THRESHOLD2,
)

print(f"Events detected: {len(stats_q)}")
stats_q.head(10)

In [ ]:
print("Peak discharge (m³/s):")
print(stats_q['peak'].describe().round(1))
print("\nDuration (days):")
print(stats_q['duration'].describe().round(1))

In [ ]:
# Hydrograph with events highlighted
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(Q_series.index, Q_series.values, lw=0.7, color="steelblue", label="Discharge")
ax.axhline(THRESHOLD, color="red", lw=1.2, ls="--", label=f"Threshold = {THRESHOLD} m³/s")
for _, row in bounds_q.iterrows():
    ax.axvspan(row["start"], row["end"], alpha=0.15, color="orange")
event_patch = mpatches.Patch(color="orange", alpha=0.4, label=f"{len(stats_q)} flood events")
ax.legend(handles=[ax.lines[0], ax.lines[1], event_patch])
ax.set_ylabel("Discharge (m³/s)")
ax.set_title("Discharge event extraction — inflection-point method", fontsize=13)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Peak vs duration scatter coloured by volume
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(stats_q["duration"], stats_q["peak"],
                c=stats_q["volume"], cmap="YlOrRd", s=60, edgecolors="k", lw=0.4)
plt.colorbar(sc, ax=ax, label="Volume (m³·s·days)")
ax.set_xlabel("Duration (days)")
ax.set_ylabel("Peak discharge (m³/s)")
ax.set_title("Flood event characteristics", fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. 🌧️ Precipitation Event Extraction

### 💡 Method: Wet-spell

1. Identify **wet days** above `threshold` (mm)
2. Merge consecutive wet runs separated by ≤ `min_gap` dry days
3. Keep only events lasting ≥ `min_duration` days
4. Extract peak intensity, total rainfall, duration, and mean intensity

In [ ]:
# Generate a synthetic daily precipitation series
prob_wet  = 0.30 + 0.15 * np.sin(2 * np.pi * (dates.dayofyear / 365 - 0.1))
wet_mask  = rng.random(n) < prob_wet
P_vals    = np.where(wet_mask, rng.exponential(8, n), 0.0)
P_series  = pd.Series(P_vals, index=dates, name="precipitation")

print(f"Wet days: {wet_mask.sum()}  ({100*wet_mask.mean():.1f} %)")
print(f"Mean    : {P_series.mean():.2f} mm/day")
print(f"Max     : {P_series.max():.1f} mm")

In [ ]:
# === PARAMETERS ===
THRESHOLD_P   = 1.0   # wet-day threshold (mm)
MIN_DURATION  = 2     # minimum spell length (days)
MIN_GAP       = 1     # dry days to bridge between consecutive wet spells

stats_p, bounds_p = extract_precipitation_events(
    series=P_series,
    threshold=THRESHOLD_P,
    min_duration=MIN_DURATION,
    min_gap=MIN_GAP,
)

print(f"Events detected: {len(stats_p)}")
stats_p.head(10)

In [ ]:
# 1-year zoom with events highlighted
YEAR = "2005"
P_yr = P_series[YEAR]
b_yr = bounds_p[(bounds_p["start"] >= P_yr.index[0]) &
                (bounds_p["end"]   <= P_yr.index[-1])]

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(P_yr.index, P_yr.values, width=1, color="royalblue", alpha=0.8)
for _, row in b_yr.iterrows():
    ax.axvspan(row["start"], row["end"], alpha=0.25, color="orange")
ax.set_ylabel("Precipitation (mm)")
ax.set_title(f"Precipitation events — {YEAR}", fontsize=13)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(stats_p["duration"], bins=20, color="royalblue", alpha=0.8, edgecolor="k")
axes[0].set_xlabel("Duration (days)")
axes[0].set_ylabel("Count")
axes[0].set_title("Event duration")
axes[0].grid(alpha=0.3)

axes[1].hist(stats_p["total"], bins=20, color="royalblue", alpha=0.8, edgecolor="k")
axes[1].set_xlabel("Total rainfall (mm)")
axes[1].set_title("Event total rainfall")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. 🔀 Unified Interface: `extract_events`

`extract_events` is a single entry point that routes to the correct method based on the `variable` argument.

In [ ]:
# Discharge — unified interface
stats_u, bounds_u = extract_events(
    series=Q_series, threshold=60.0, variable="discharge", threshold2=45.0
)
print(f"Discharge events: {len(stats_u)}")

# Precipitation — unified interface
stats_up, bounds_up = extract_events(
    series=P_series, threshold=1.0, variable="precipitation",
    min_duration=2, min_gap=1,
)
print(f"Precipitation events: {len(stats_up)}")